# R2AI2026 — sinh `pandas_query` trên Kaggle (T4 GPU)

Điều kiện trước khi chạy:
1. Bật **GPU T4** (Settings → Accelerator) và **Internet** (để `git clone` + tải model).
2. Upload `retrieval_results.jsonl` (build ở local bằng `python -m r2ai.retrieval.run_retrieval`) làm **Kaggle Dataset**, rồi Add Data vào notebook. File này đã nhúng sẵn CSV của các bảng candidate nên **không cần mount corpus 362MB**.
3. Sửa `REPO_URL` và `RETRIEVAL_PATH` bên dưới cho khớp.

Output: `predictions.jsonl` ghi **append + flush sau mỗi câu** trong `/kaggle/working` — tải về rồi chạy `python -m r2ai.packaging.assemble_submission` ở local (re-execute lại toàn bộ query trước khi đóng gói).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/r2ai-stage2.git"
RETRIEVAL_PATH = "/kaggle/input/datasets/nhtquyn/r2ai2026/retrieval_results.jsonl"
PREDICTIONS_PATH = "/kaggle/working/predictions.jsonl"
WORK_DIR = "/kaggle/working/exec"
PILOT_N = 20  # chạy thử trước khi chạy full 1.012 câu

In [ ]:
!git clone --depth 1 $REPO_URL /kaggle/working/r2ai-stage2
%cd /kaggle/working/r2ai-stage2
# Cài đúng theo requirements-kaggle.txt (nó `-r requirements.txt` nên ghim luôn pandas/numpy —
# bắt buộc, vì đáp án cuối được re-execute ở local: lệch version pandas/numpy giữa 2 nơi thì
# số liệu exec_ok của pilot ở đây sẽ gây hiểu lầm).
# KHÔNG cài lại torch: image Kaggle đã có bản khớp CUDA.
!pip install -q -r requirements-kaggle.txt

In [ ]:
import sys

sys.path.insert(0, "/kaggle/working/r2ai-stage2")
import numpy
import pandas
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), (
    "Không phát hiện GPU! Vào Settings (panel bên phải notebook) -> Accelerator -> chọn GPU T4 x2 "
    "(hoặc T4 x1) -> Save -> notebook sẽ restart session. Chạy lại từ đầu sau đó. Không có GPU thì "
    "load model 7B sẽ cực chậm/OOM trên CPU."
)

# Version phải khớp requirements.txt, nếu không thì kết quả exec ở đây sẽ lệch với lúc
# re-execute ở local (bước assemble_submission). Đây chỉ là cảnh báo thông tin — KHÔNG ảnh hưởng
# đáp án nộp cuối, vì assemble_submission luôn re-execute lại toàn bộ query ở local trước khi đóng gói.
EXPECTED = {"pandas": "2.2.3", "numpy": "2.1.3"}
actual = {"pandas": pandas.__version__, "numpy": numpy.__version__}
print("version thực tế:", actual)
for name, want in EXPECTED.items():
    if actual[name] != want:
        print(f"  ⚠️  {name} {actual[name]} != {want} ghim trong requirements.txt — bản Kaggle khác "
              f"local, chỉ ảnh hưởng số liệu pilot ở notebook này, không ảnh hưởng đáp án nộp cuối.")

# Fail sớm ngay tại đây nếu cell cài đặt ở trên bị lỗi (vd: 1 package trong requirements-kaggle.txt
# không có wheel sẵn cho Python của image Kaggle -> pip cố build từ source -> thiếu toolchain ->
# TOÀN BỘ lệnh `pip install -r requirements-kaggle.txt` crash nhưng !pip không làm cell fail, nên
# lỗi trước đây chỉ lộ ra rất muộn, ngay lúc load model 7B ở bước pilot) — kiểm tra ngay từ đây thay
# vì để lỗi rơi xuống sâu, khó truy nguyên.
import transformers
import bitsandbytes

print("transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__)

In [ ]:
# Đăng nhập HuggingFace Hub (không bắt buộc với Qwen, chỉ để bớt cảnh báo rate-limit khi tải model).
# Điền token thật vào đây SAU KHI đã import notebook này lên Kaggle — đừng commit bản đã điền token
# ngược lại về git.
from huggingface_hub import login

login("HF_TOKEN")  # TODO: dán token HuggingFace thật vào đây trên Kaggle

## 1. Smoke test: prompt + sandbox (không cần GPU)

`--dry-run` không nạp LLM: chỉ dựng prompt, ghi CSV ra đĩa và chạy sandbox. Bắt sớm lỗi encode CSV / đường dẫn trước khi tốn thời gian GPU.

In [ ]:
!python -m r2ai.generation.run_generation \
    --retrieval $RETRIEVAL_PATH \
    --out /kaggle/working/predictions_dryrun.jsonl \
    --work-dir $WORK_DIR --limit 3 --dry-run --no-resume

import subprocess
import time

start = time.time()
proc = subprocess.run([
    "python", "-m", "r2ai.generation.run_generation",
    "--retrieval", RETRIEVAL_PATH, "--out", PREDICTIONS_PATH, "--work-dir", WORK_DIR,
    "--limit", str(PILOT_N),
])
elapsed = time.time() - start

if proc.returncode != 0:
    print(f"❌ Pilot THẤT BẠI (exit code {proc.returncode}) sau {elapsed:.0f}s — sửa lỗi ở traceback "
          "phía trên rồi chạy lại cell này. Số liệu thời gian dưới đây (nếu có) là VÔ NGHĨA, đừng tin.")
else:
    print(f"{elapsed / PILOT_N:.1f}s/câu -> ước tính {elapsed / PILOT_N * 1012 / 3600:.1f}h cho 1.012 câu")

In [ ]:
import time

start = time.time()
!python -m r2ai.generation.run_generation \
    --retrieval $RETRIEVAL_PATH --out $PREDICTIONS_PATH --work-dir $WORK_DIR --limit $PILOT_N
elapsed = time.time() - start
print(f"{elapsed / PILOT_N:.1f}s/câu -> ước tính {elapsed / PILOT_N * 1012 / 3600:.1f}h cho 1.012 câu")

## 3. Chạy full (resume được)

Mặc định `--resume`: bỏ qua các id đã có trong `predictions.jsonl`, nên chạy lại cell này sau khi session bị ngắt là tiếp tục từ chỗ dừng.

In [ ]:
!python -m r2ai.generation.run_generation \
    --retrieval $RETRIEVAL_PATH --out $PREDICTIONS_PATH --work-dir $WORK_DIR

In [ ]:
import json
from collections import Counter

rows = [json.loads(line) for line in open(PREDICTIONS_PATH, encoding="utf-8") if line.strip()]
print("tổng:", len(rows), "| id duy nhất:", len({r["id"] for r in rows}))
print("exec_ok:", Counter(r["exec_ok"] for r in rows))
for row in [r for r in rows if not r["exec_ok"]][:5]:
    print(row["id"], "->", (row["exec_error"] or "")[:200])

Tải `predictions.jsonl` về máy local, đặt vào `data/interim/`, rồi:

```bash
python -m r2ai.packaging.assemble_submission   # join + re-execute ở local
python -m r2ai.packaging.zip_submission        # validate + zip
```